In [290]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error

In [291]:
dataset = pd.read_csv('/kaggle/input/datasets/franklinflores10/titanicdts/titanic.csv')
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [292]:
#DataCleaning
dataset.dropna(subset = 'Embarked', inplace = True)
dataset.drop(columns = 'Cabin', inplace = True)
median_age = dataset.Age.median()
median_age
dataset.Age = dataset.Age.fillna(dataset.Age.median())

#Morphing Data for Models
dataset = pd.get_dummies(dataset, columns = ['Sex', 'Embarked'])

In [293]:
dataset.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Sex_female     0
Sex_male       0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
dtype: int64

In [294]:
dataset.describe()
dataset.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q',
       'Embarked_S'],
      dtype='object')

In [295]:
dataset_features = ['Pclass', 'Sex_male','Sex_female', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
X = dataset[dataset_features]
y = dataset.Survived

train_X, val_X, train_y, val_y = train_test_split(X,y, random_state = 0)

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    titanic_model = RandomForestClassifier(max_leaf_nodes = max_leaf_nodes, n_estimators = 100, random_state = 0)
    titanic_model.fit(train_X,train_y)
    titanic_predictions = titanic_model.predict(val_X)
    mae = mean_absolute_error(val_y,titanic_predictions)
    return(mae)

#testing_max_leaf_nodes = [5,50,500,5000,50000]
#testing_max_leaf_nodes = [30,35,40,45,50,55,60,65]
testing_max_leaf_nodes = [30,31,32,33,34,35,36,37,38,39,40]
MAE_values = []
for max_leaf_nodes in testing_max_leaf_nodes:
    current_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    MAE_values.append(current_mae)

position = MAE_values.index(min(MAE_values))
best_tree_size = testing_max_leaf_nodes[position]
print(best_tree_size)


31


In [296]:
# Best max_leaf_node = 31
titanic_model = RandomForestClassifier(max_leaf_nodes = best_tree_size, n_estimators = 100, random_state = 0)
titanic_model.fit(train_X,train_y)
titanic_predictions = titanic_model.predict(val_X)
mae = mean_absolute_error(val_y,titanic_predictions)

In [297]:
print(f"Max Leaf Nodes: {best_tree_size}\n\nMean Absolute Error: {mae}")
print(f"\nPredictions: {titanic_predictions.tolist()}")
print(f"\nActual Results: {val_y.tolist()}")

Max Leaf Nodes: 31

Mean Absolute Error: 0.21076233183856502

Predictions: [1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1]

Actual Results: [0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 